# 12.3 Numeric options for display

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The numbers in a `table` or list produced by `display` are the result of a transformation
from the computer's internal numeric representation to a string of digits and symbols.
AMPL's options for adjusting this transformation are shown in [Table 12-2](./12_3_numeric_options_for_display.ipynb#`table`-12-2). In this
section we first consider options that affect only the appearance of numbers, and then
options that affect underlying solution values as well.

```ampl
display_eps         smallest magnitude displayed differently from zero (0)
display_precision   digits of precision to which displayed numbers are rounded; full precision
					if 0 (6)
display_round       digits left or (if negative) right of decimal place to which displayed
					numbers are rounded, overriding display_precision ("")
solution_precision  digits of precision to which solution values are rounded; full precision
					if 0 (0)
solution_round      digits left or (if negative) right of decimal place to which solution
					values are rounded, overriding solution_precision ("")
```

**[Table 12-2](./12_3_numeric_options_for_display.ipynb#`table`-12-2):** Numeric options for `display` (with default values).

**Appearance of numeric values**

In all of our examples so far, the `display` command shows each numerical value to
the same number of significant digits:

```ampl
ampl: display {p in PROD, t in 1..T} Make[p,t]/rate[p];
Make[p,t]/rate[p] [*,*] (tr)
:    bands     coils      :=
1   29.95     10.05
2   30        10
3   20        12
4   32.1429    7.85714
;
ampl: display {p in PROD, t in 1..T} prodcost[p]*Make[p,t];
prodcost[p]*Make[p,t] [*,*] (tr)
:    bands    coils    :=
1   59900     15477
2   60000     15400
3   40000     18480
4   64285.7   12100
;
```

(see Figures 6-3 and 6-4). The default is to use six significant digits, whether the result
comes out as 7.85714 or 64285.7. Some numbers seem to have fewer digits, but only
because trailing zeros have been dropped; 29.95 represents the number that is
exactly 29.9500 to six digits, for example, and 59900 represents 59900.0.

By changing the option display_precision to a value other than six, you can
vary the number of significant digits reported:

```ampl
ampl: option display_precision 3;
ampl: display Make['bands',4] / rate['bands'],
ampl?    prodcost['bands'] * Make['bands',4];
Make['bands',4]/rate['bands'] = 32.1
prodcost['bands']*Make['bands',4] = 64300
ampl: option display_precision 9;
ampl: display Make['bands',4] / rate['bands'],
ampl?    prodcost['bands'] * Make['bands',4];
Make['bands',4]/rate['bands'] = 32.1428571
prodcost['bands']*Make['bands',4] = 64285.7143
ampl: option display_precision 0;
ampl: display Make['bands',4] / rate['bands'],
ampl?    prodcost['bands'] * Make['bands',4];
Make['bands',4]/rate['bands'] = 32.14285714285713
prodcost['bands']*Make['bands',4] = 64285.71428571427
```

In the last example, a display_precision of 0 is interpreted specially; it tells
`display` to represent numbers as exactly as possible, using however many digits are
necessary. (To be precise, the displayed number is the shortest decimal representation
that, when correctly rounded to the computer's representation, gives the value exactly as
stored in the computer.)
Displays to a given precision provide the same degree of useful information about
each number, but they can look ragged due to the varying numbers of digits after the decimal
point. To specify rounding to a fixed number of decimal places, regardless of the
resulting precision, you may set the option display_round. A non negative value
specifies the number of digits to appear after the decimal point:

```ampl
ampl: option display_round 2;
ampl: display {p in PROD, t in 1..T} Make[p,t]/rate[p];
Make[p,t]/rate[p] [*,*] (tr)
:   bands   coils    :=
1   29.95   10.05
2   30.00   10.00
3   20.00   12.00
4   32.14    7.86
;
```

A negative value indicates rounding before the decimal point. For example, when
display_round is -2, all numbers are rounded to hundreds:

```ampl
ampl: option display_round -2;
ampl: display {p in PROD, t in 1..T} prodcost[p]*Make[p,t];
prodcost[p]*Make[p,t] [*,*] (tr)
:   bands   coils    :=
1   59900   15500
2   60000   15400
3   40000   18500
4   64300   12100
;
```

Any `integer` value of display_round overrides the effect of display_precision.
To turn off display_round, set it to some non-`integer` such as the empty string ".

Depending on the solver you employ, you may find that some of the solution values
that ought to be zero do not always quite come out that way. For example, here is one
solver's report of the `objective` function terms cost[i,j] * Trans[i,j] for the
assignment problem of Section 3.3:

```ampl
ampl: option omit_zero_rows 1;
ampl: display {i in ORIG, j in DEST} cost[i,j] * Trans[i,j];
cost[i,j]*Trans[i,j] :=
Coullard C118        6
Coullard D241        2.05994e-17
Daskin     D237      1
Hazen      C246      1
Hopp       D237      6.86647e-18
Hopp       D241      4
... 9 lines omitted
White      C246      2.74659e-17
White      C251     -3.43323e-17
White      M239      1
;
```

Minuscule values like 6.86647e-18 and -3.43323e-17 have no significance in the context
of this problem; they would be zeros in an exact solution, but come out slightly nonzero
as an artifact of the way that the solver's algorithm interacts with the computer's representation
of numbers.

To avoid viewing these numbers in meaningless precision, you can pick a reasonable
setting for display_round — in this case 0, since there are no digits of interest after
the decimal point:

```ampl
ampl: option display_round 0;
ampl: display {i in ORIG, j in DEST} cost[i,j] * Trans[i,j];
cost[i,j]*Trans[i,j] :=
Coullard C118     6
Coullard D241     0
Daskin    D237    1
Hazen     C246    1
Hopp      D237    0
Hopp      D241    4
Iravani   C118    0
Iravani   C138    2
Linetsky C250     3
Mehrotra D239     2
Nelson    C138    0
Nelson    C140    4
Smilowitz M233    1
Tamhane   C118   -0
Tamhane   C251    3
White     C246    0
White     C251   -0
White     M239    1
;
```

The small numbers are now represented only as 0 if positive or -0 if negative. If you
want to suppress their appearance entirely, however, you must set a separate option,
display_eps:

```ampl
ampl: option display_eps 1e-10;
ampl: display {i in ORIG, j in DEST} cost[i,j] * Trans[i,j];
cost[i,j]*Trans[i,j] :=
Coullard C118    6
Daskin    D237   1
Hazen     C246   1
Hopp      D241   4
Iravani   C138   2
Linetsky C250    3
Mehrotra D239    2
Nelson    C140   4
Smilowitz M233   1
Tamhane   C251   3
White     M239   1
;
```

Any value whose magnitude is less than the value of display_eps is treated as an
exact zero in all output of `display`.

Rounding of solution values
The options display_precision, display_round and display_eps affect
only the appearance of numbers, not their actual values. You can see this if you try to
`display` the set of all pairs of `i` in ORIG and `j` in DEST that have a positive value in the
preceding `table`, by comparing cost[i,j]*Trans[i,j] to 0:

```ampl
ampl: display {i in ORIG, j in DEST: cost[i,j]*Trans[i,j] > 0};
set {i in ORIG, j in DEST: cost[i,j]*Trans[i,j] > 0} :=
(Coullard,C118)    (Iravani,C118)     (Smilowitz,M233)
(Coullard,D241)    (Iravani,C138)     (Tamhane,C251)
(Daskin,D237)      (Linetsky,C250)    (White,C246)
(Hazen,C246)       (Mehrotra,D239)    (White,M239)
(Hopp,D237)        (Nelson,C138)
(Hopp,D241)        (Nelson,C140);
```

Even though a value like 2.05994e-17 is treated as a zero for purposes of `display`, it
tests greater than zero. You could `fix` this problem by changing > 0 above to, say, > 0.1.
As an alternative, you can set the option solution_round so that AMPL rounds the
solution values to a reasonable precision when they are received from the solver:

```ampl
ampl: option solution_round 10;
ampl: solve;
MINOS 5.5: optimal solution found.

40 iterations, objective 28
ampl: display {i in ORIG, j in DEST: cost[i,j]*Trans[i,j] > 0};
set {i in ORIG, j in DEST: cost[i,j]*Trans[i,j] > 0} :=
(Coullard,C118)    (Iravani,C138)     (Smilowitz,M233)
(Daskin,D237)      (Linetsky,C250)    (Tamhane,C251)
(Hazen,C246)       (Mehrotra,D239)    (White,M239)
(Hopp,D241)        (Nelson,C140);
```

The options solution_precision and solution_round work in the same way
as display_precision and display_round, except that they are applied only to
solution values upon return from a solver, and they permanently change the returned values
rather than only their appearance.

Rounded values can make a difference even when they are not near zero. As an
example, we first use several `display` options to get a compact listing of the fractional
solution to the scheduling `model` of [Figure 16-4](#missing) <!--- xTODO missing reference --->:

```ampl
ampl: model sched.mod;
ampl: data sched.dat;
ampl: solve;
MINOS 5.5: optimal solution found.

19 iterations, objective 265.6
```

```ampl
ampl: option display_width 60;
ampl: option display_1col 5;
ampl: option display_eps 1e-10;
ampl: option omit_zero_rows 1;
ampl: display Work;
Work [*] :=
 10 28.8   30 14.4    71 35.6    106 23.2   123 35.6
 18 7.6    35 6.8     73 28      109 14.4
 24 6.8    66 35.6    87 14.4    113 14.4
;
```

Each value Work[j] represents the number of workers assigned to schedule j. We can
get a quick practical schedule by rounding the fractional values up to the next highest
`integer`; using the ceil function to perform the rounding, we see that the total number of
workers needed should be:

```ampl
ampl: display sum {j in SCHEDS} ceil(Work[j]);
sum{j in SCHEDS} ceil(Work[j]) = 273
```

If we copy the numbers from the preceding `table` and round them up by hand, however,
we find that they only sum to 271. The source of the difficulty can be seen by displaying
the numbers to full precision:

```ampl
ampl: option display_eps 0;
ampl: option display_precision 0;
ampl: display Work;
Work [*] :=
 10 28.799999999999997          73   28.000000000000018
 18 7.599999999999998           87   14.399999999999995
 24 6.79999999999999            95   -5.876671973951407e-15
 30 14.40000000000001          106   23.200000000000006
 35 6.799999999999995          108    4.685288280240683e-16
 55 -4.939614313857677e-15     109   14.4
 66 35.6                       113   14.4
 71 35.599999999999994         123   35.59999999999999
;
```

Half the problem is due to the minuscule positive value of Work[108], which was
rounded up to 1. The other half is due to Work[73]; although it is 28 in an exact solution,
it comes back from the solver with a slightly larger value of 28.000000000000018,
so it gets rounded up to 29.

The easiest way to ensure that our arithmetic works correctly in this case is again to
set solution_round before `solve`:

```ampl
ampl: option solution_round 10;
ampl: solve;
MINOS 5.5: optimal solution found.

19 iterations, objective 265.6
ampl: display sum {j in SCHEDS} ceil(Work[j]);
sum{j in SCHEDS} ceil(Work[j]) = 271
```

We picked a value of 10 for solution_round because we observed that the slight
inaccuracies in the solver's values occurred well past the 10th decimal place.

The effect of solution_round or solution_precision applies to all values
returned by the solver. To modify only certain values, use the assignment (`let`) command
described in [Section 11.3](../11/11_3_modifying_data.ipynb) together with the rounding functions in [Table 11-1](../11/11_3_modifying_data.ipynb#`table`-11-1).